In [ ]:
# Path config (auto-injected). Loads WORK_DIR from .env so notebooks run anywhere.
import os
from pathlib import Path
from dotenv import load_dotenv

_p = Path.cwd().resolve()
for _ in range(8):
    if (_p / '.env').is_file():
        load_dotenv(_p / '.env')
        break
    if _p == _p.parent:
        break
    _p = _p.parent

WORK_DIR = os.environ['WORK_DIR']
RETRIEVAL_DIR = f"{WORK_DIR}/retrieval_assets"

# 12 - Natural Hallucination Test (Degraded Retrieval)

## Purpose
Test our detector on **naturally occurring** hallucinations, not prompt-injected ones.

## Why This Matters
Our current dataset uses prompt-injection to create hallucinations ("change one fact"). This creates **obvious** errors that perplexity easily catches.

Real RAG hallucinations happen when:
- Context is insufficient (model fills gaps)
- Model misinterprets ambiguous context
- Model confidently blends correct + incorrect info

## Method
1. Take 100 HotpotQA questions
2. **Degrade retrieval** (k=1 instead of k=5, or add noise)
3. Let model generate naturally (NO prompt injection)
4. Output for manual labeling
5. Run our detector on labeled examples

## Expected Outcome
- AUROC will likely drop (0.65-0.75 range)
- But if it still beats baselines, that's a **strong** result
- If it fails, we have an honest limitation to discuss

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## Step 1: Setup

In [ ]:
import torch
print(f"GPU Available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")

In [ ]:
!pip install -q transformers accelerate bitsandbytes sentence-transformers datasets faiss-cpu

In [ ]:
import os
import json
import faiss
import numpy as np
import torch
from datasets import load_dataset
from sentence_transformers import SentenceTransformer
from transformers import AutoModelForCausalLM, AutoTokenizer
from tqdm import tqdm
import pickle

## Step 2: Load Model

In [ ]:
from huggingface_hub import login
login(token=os.environ['HF_TOKEN'])  # Paste your token here


In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM
import torch
# Load Llama-3.1-8B-Instruct
model_name = "meta-llama/Llama-3.1-8B-Instruct"

print(f"Loading {model_name}...")

model = AutoModelForCausalLM.from_pretrained(
    model_name,
    torch_dtype=torch.float16,
    device_map="auto",
    trust_remote_code=True
)

tokenizer = AutoTokenizer.from_pretrained(model_name)
tokenizer.pad_token = tokenizer.eos_token

print(f"✓ Model loaded successfully")
print(f"✓ Device: {model.device}")
print(f"✓ Memory allocated: {torch.cuda.memory_allocated() / 1e9:.2f} GB")


print(f"\nModel loaded!")

print(f"✓ Loaded {model_name}")
print(f"✓ GPU: {torch.cuda.get_device_name(0)}")
print(f"✓ GPU Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")


print(f"\nModel loaded!")

print(f"✓ Loaded {model_name}")
print(f"✓ GPU: {torch.cuda.get_device_name(0)}")
print(f"✓ GPU Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")


## Step 3: Load FAISS Index and Embedder

In [ ]:
embedder = SentenceTransformer("all-MiniLM-L6-v2")

# Load HotpotQA index
# Update path for Kaggle if needed
INDEX_PATH = f"{RETRIEVAL_DIR}/hotpot_faiss.index"  # Kaggle path
CORPUS_PATH = f"{RETRIEVAL_DIR}/hotpot_corpus.json"  # Kaggle path

# Alternative local paths
# INDEX_PATH = "hotpot_faiss.index"
# CORPUS_PATH = "hotpot_corpus.json"

index = faiss.read_index(INDEX_PATH)
with open(CORPUS_PATH) as f:
    corpus = json.load(f)

# Load questions
qa_dataset = load_dataset("hotpot_qa", "distractor", split="validation")

print(f"Loaded {index.ntotal} vectors")
print(f"Loaded {len(corpus)} paragraphs")
print(f"Loaded {len(qa_dataset)} questions")

## Step 4: Define Generation Functions

In [ ]:
def retrieve(query, k=5):
    """Retrieve top-k documents"""
    q_emb = embedder.encode([query])
    faiss.normalize_L2(q_emb)
    _, indices = index.search(np.array(q_emb), k)
    return [corpus[i] for i in indices[0]]


def retrieve_degraded(query, mode="k1"):
    """
    Retrieve with degraded quality to induce natural hallucinations.

    Modes:
    - 'k1': Only retrieve 1 document (insufficient context)
    - 'k1_offset': Retrieve 1 document, but skip the best match
    - 'noisy': Retrieve k=5 but replace 3 with random docs
    """
    q_emb = embedder.encode([query])
    faiss.normalize_L2(q_emb)

    if mode == "k1":
        # Only get top-1 document
        _, indices = index.search(np.array(q_emb), 1)
        return [corpus[indices[0][0]]]

    elif mode == "k1_offset":
        # Get documents ranked 3-5 (skip best matches)
        _, indices = index.search(np.array(q_emb), 5)
        return [corpus[indices[0][3]]]  # Use 4th best match

    elif mode == "noisy":
        # Get top-2 relevant + 3 random documents
        _, indices = index.search(np.array(q_emb), 2)
        relevant = [corpus[i] for i in indices[0]]

        # Add 3 random documents
        random_indices = np.random.choice(len(corpus), 3, replace=False)
        random_docs = [corpus[i] for i in random_indices]

        # Shuffle to mix relevant and random
        all_docs = relevant + random_docs
        np.random.shuffle(all_docs)
        return all_docs

    else:
        raise ValueError(f"Unknown mode: {mode}")


def generate_answer(question, context, max_new_tokens=100):
    """
    Generate answer naturally - NO prompt injection!
    Just ask the model to answer based on context.
    """
    if isinstance(context, list):
        context_str = "\n\n".join(context)
    else:
        context_str = context

    prompt = f"""Answer the question based on the context below. Be concise (1-2 sentences).

Context:
{context_str}

Question: {question}

Answer:"""

    messages = [{"role": "user", "content": prompt}]

    input_text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )

    inputs = tokenizer(input_text, return_tensors="pt").to("cuda")

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            temperature=0.3,  # Some variability, but not too much
            do_sample=True,
            pad_token_id=tokenizer.eos_token_id
        )

    response = tokenizer.decode(outputs[0][inputs['input_ids'].shape[1]:], skip_special_tokens=True)
    return response.strip()


def extract_uncertainty(question, context, answer):
    """
    Extract uncertainty metrics for the generated answer.
    Returns: perplexity, mean_prob, min_prob
    """
    if isinstance(context, list):
        context_str = "\n\n".join(context)
    else:
        context_str = context

    prompt = f"""Answer the question based on the context below.

Context:
{context_str}

Question: {question}

Answer:"""

    messages = [{"role": "user", "content": prompt}]

    input_text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )

    # Encode prompt + answer
    full_text = input_text + answer
    inputs = tokenizer(full_text, return_tensors="pt").to("cuda")
    prompt_ids = tokenizer(input_text, return_tensors="pt").to("cuda")
    prompt_len = prompt_ids['input_ids'].shape[1]

    with torch.no_grad():
        outputs = model(**inputs)
        logits = outputs.logits

    # Get probabilities for answer tokens only
    answer_logits = logits[0, prompt_len-1:-1, :]  # Shift by 1 for next-token prediction
    answer_tokens = inputs['input_ids'][0, prompt_len:]

    probs = torch.softmax(answer_logits.float(), dim=-1)

    token_probs = probs[range(len(answer_tokens)), answer_tokens]

    # Compute metrics
    token_probs_np = token_probs.cpu().numpy()

    if len(token_probs_np) == 0:
        return 1.0, 1.0, 1.0  # Fallback

    mean_prob = float(np.mean(token_probs_np))
    min_prob = float(np.min(token_probs_np))

    # Perplexity = exp(-mean(log(p)))
    log_probs = np.log(token_probs_np + 1e-10)
    perplexity = float(np.exp(-np.mean(log_probs)))

    return perplexity, mean_prob, min_prob


print("Functions defined!")

## Step 5: Generate Natural Hallucination Test Set

We generate answers with **degraded retrieval** and output for manual labeling.

In [ ]:
# Configuration
N_SAMPLES = 100
DEGRADATION_MODE = "k1"  # Options: "k1", "k1_offset", "noisy"

print(f"Generating {N_SAMPLES} samples with degraded retrieval (mode: {DEGRADATION_MODE})...")
print("="*60)

natural_test_samples = []

# Use validation set (different from training)
indices = np.random.choice(len(qa_dataset), N_SAMPLES, replace=False)

for i, idx in enumerate(tqdm(indices)):
    item = qa_dataset[int(idx)]
    question = item['question']
    gold_answer = item['answer']  # Ground truth for reference

    # Get degraded context
    context = retrieve_degraded(question, mode=DEGRADATION_MODE)

    # Generate answer naturally (NO injection!)
    answer = generate_answer(question, context)

    # Extract uncertainty metrics
    perplexity, mean_prob, min_prob = extract_uncertainty(question, context, answer)

    # Compute SDS
    context_emb = embedder.encode([' '.join(context) if isinstance(context, list) else context])[0]
    answer_emb = embedder.encode([answer])[0]
    sds = float(np.linalg.norm(answer_emb - context_emb))

    sample = {
        "id": i,
        "question": question,
        "gold_answer": gold_answer,
        "context": context,
        "generated_answer": answer,
        "degradation_mode": DEGRADATION_MODE,
        # Metrics (for later evaluation)
        "perplexity": perplexity,
        "mean_prob": mean_prob,
        "min_prob": min_prob,
        "sds": sds,
        # TO BE FILLED BY HUMAN
        "hallucinated": None  # 0 = correct, 1 = hallucinated
    }

    natural_test_samples.append(sample)

    # Save checkpoint every 20
    if (i + 1) % 20 == 0:
        with open(f'natural_hallucination_checkpoint_{DEGRADATION_MODE}.json', 'w') as f:
            json.dump(natural_test_samples, f, indent=2)

print(f"\nGenerated {len(natural_test_samples)} samples!")

## Step 6: Save for Manual Labeling

In [ ]:
# Save full data (with metrics) for later evaluation
output_file = f'natural_hallucination_test_{DEGRADATION_MODE}.json'
with open(output_file, 'w') as f:
    json.dump(natural_test_samples, f, indent=2)

print(f"Saved full data to: {output_file}")

# Create human-readable labeling sheet
print("\n" + "="*80)
print("MANUAL LABELING SHEET")
print("="*80)
print("\nInstructions:")
print("- Label each answer as 0 (CORRECT) or 1 (HALLUCINATED)")
print("- Hallucinated = contains ANY factual error, unsupported claim, or fabrication")
print("- Compare to gold answer AND check if claims are supported by context")
print("="*80 + "\n")

In [ ]:
# Create CSV for easy labeling
import csv

csv_file = f'natural_hallucination_labeling_{DEGRADATION_MODE}.csv'

with open(csv_file, 'w', newline='', encoding='utf-8') as f:
    writer = csv.writer(f)
    writer.writerow(['ID', 'Question', 'Gold Answer', 'Generated Answer', 'Context (truncated)', 'Label (0=correct, 1=hallucinated)'])

    for sample in natural_test_samples:
        context_preview = sample['context'][0][:200] + "..." if sample['context'] else ""
        writer.writerow([
            sample['id'],
            sample['question'],
            sample['gold_answer'],
            sample['generated_answer'],
            context_preview,
            ''  # To be filled
        ])

print(f"Saved labeling sheet to: {csv_file}")
print("\nOpen this CSV in Excel/Google Sheets and fill the 'Label' column.")

## Step 7: Preview Some Examples

In [ ]:
print("\n" + "="*80)
print("SAMPLE EXAMPLES FOR LABELING")
print("="*80)

for i, sample in enumerate(natural_test_samples[:10]):
    print(f"\n--- Example {i+1} ---")
    print(f"Question: {sample['question']}")
    print(f"Gold Answer: {sample['gold_answer']}")
    print(f"Generated: {sample['generated_answer']}")
    print(f"Context (first 200 chars): {sample['context'][0][:200]}...")
    print(f"Perplexity: {sample['perplexity']:.3f}, Mean Prob: {sample['mean_prob']:.3f}")
    print(f"Label: _____ (fill in 0 or 1)")

## Step 8: After Labeling - Load Labels and Evaluate

**Run this section AFTER you've manually labeled the CSV file.**

In [ ]:
# ============================================================
# AFTER MANUAL LABELING: Load labels and evaluate
# ============================================================

# Option 1: Load from labeled CSV
LABELED_CSV = f'natural_hallucination_labeling_{DEGRADATION_MODE}_LABELED.csv'

# Option 2: Load from JSON with labels filled in
LABELED_JSON = f'natural_hallucination_test_{DEGRADATION_MODE}_LABELED.json'

# Try JSON first
try:
    with open(LABELED_JSON) as f:
        labeled_samples = json.load(f)
    print(f"Loaded labels from {LABELED_JSON}")
except FileNotFoundError:
    # Try CSV
    try:
        labeled_samples = natural_test_samples.copy()
        with open(LABELED_CSV, 'r', encoding='utf-8') as f:
            reader = csv.DictReader(f)
            for row in reader:
                idx = int(row['ID'])
                label = row.get('Label (0=correct, 1=hallucinated)', '').strip()
                if label in ['0', '1']:
                    labeled_samples[idx]['hallucinated'] = int(label)
        print(f"Loaded labels from {LABELED_CSV}")
    except FileNotFoundError:
        print("ERROR: No labeled file found!")
        print(f"Please label {csv_file} and save as {LABELED_CSV}")
        print("Or fill 'hallucinated' field in JSON and save as {LABELED_JSON}")
        labeled_samples = None

In [ ]:
# Evaluate if labels exist
if labeled_samples and all(s.get('hallucinated') is not None for s in labeled_samples):
    from sklearn.metrics import roc_auc_score, classification_report

    labels = np.array([s['hallucinated'] for s in labeled_samples])
    perplexity = np.array([s['perplexity'] for s in labeled_samples])
    mean_prob = np.array([s['mean_prob'] for s in labeled_samples])
    sds = np.array([s['sds'] for s in labeled_samples])

    print("\n" + "="*60)
    print("EVALUATION ON NATURAL HALLUCINATIONS")
    print("="*60)

    print(f"\nDataset Statistics:")
    print(f"  Total samples: {len(labels)}")
    print(f"  Hallucinated: {sum(labels)} ({100*sum(labels)/len(labels):.1f}%)")
    print(f"  Correct: {len(labels) - sum(labels)} ({100*(len(labels)-sum(labels))/len(labels):.1f}%)")

    print("\n" + "-"*60)
    print("AUROC Results:")
    print("-"*60)

    # Perplexity (higher = more likely hallucinated)
    auroc_perp = roc_auc_score(labels, perplexity)
    print(f"  Perplexity:           {auroc_perp:.4f}")

    # Mean prob (lower = more likely hallucinated, so negate)
    auroc_prob = roc_auc_score(labels, -mean_prob)
    print(f"  Mean Prob (neg):      {auroc_prob:.4f}")

    # SDS (higher = more likely hallucinated)
    auroc_sds = roc_auc_score(labels, sds)
    print(f"  SDS:                  {auroc_sds:.4f}")

    # Combined (simple average)
    combined = (perplexity / perplexity.max()) + (sds / sds.max()) - (mean_prob / mean_prob.max())
    auroc_combined = roc_auc_score(labels, combined)
    print(f"  Combined:             {auroc_combined:.4f}")

    print("\n" + "-"*60)
    print("Comparison to Prompt-Injected Results:")
    print("-"*60)
    print(f"  Prompt-injected AUROC: 0.809 (SDS + Uncertainty)")
    print(f"  Natural AUROC:         {auroc_combined:.4f}")
    print(f"  Difference:            {auroc_combined - 0.809:+.4f}")

    if auroc_combined >= 0.70:
        print("\n  RESULT: Method still works on natural hallucinations!")
    elif auroc_combined >= 0.60:
        print("\n  RESULT: Moderate performance - discuss as limitation")
    else:
        print("\n  RESULT: Poor performance on natural hallucinations - important finding!")
else:
    print("Labels not yet filled in. Please complete manual labeling first.")

## Step 9: Save Final Labeled Dataset

In [ ]:
if labeled_samples and all(s.get('hallucinated') is not None for s in labeled_samples):
    # Save the fully labeled dataset
    output_file = f'natural_hallucination_FINAL_{DEGRADATION_MODE}.json'
    with open(output_file, 'w') as f:
        json.dump(labeled_samples, f, indent=2)
    print(f"Saved final labeled dataset to: {output_file}")

    # Also save just the metrics for easy analysis
    metrics_file = f'natural_hallucination_metrics_{DEGRADATION_MODE}.pkl'
    metrics = {
        'labels': labels,
        'perplexity': perplexity,
        'mean_prob': mean_prob,
        'sds': sds,
        'combined': combined,
        'auroc_perplexity': auroc_perp,
        'auroc_mean_prob': auroc_prob,
        'auroc_sds': auroc_sds,
        'auroc_combined': auroc_combined
    }
    with open(metrics_file, 'wb') as f:
        pickle.dump(metrics, f)
    print(f"Saved metrics to: {metrics_file}")

## Summary

### What This Notebook Does:
1. Generates 100 QA examples with **degraded retrieval** (k=1 instead of k=5)
2. Model generates answers **naturally** (no prompt injection)
3. Extracts uncertainty metrics (perplexity, mean_prob, SDS)
4. Outputs CSV for manual labeling
5. Evaluates your detector on **realistic** hallucinations

### Expected Results:
- With degraded retrieval, expect 40-60% hallucination rate
- AUROC will likely be lower than 0.809 (prompt-injected)
- If AUROC > 0.70: Strong evidence method works on real cases
- If AUROC < 0.60: Important limitation to discuss in paper

### For Paper:
This addresses the critique: "Your hallucinations are unrealistic"

You can write:
> "To validate our method on naturally occurring hallucinations, we evaluated on 100 examples generated with degraded retrieval (k=1). The model achieved [X] AUROC, demonstrating [the method's robustness / an important limitation of uncertainty-based detection]."